In [ ]:
# Import packages
#!pip install pandas stingray

import os
import sys
import pyvo
import glob
import shutil
import stingray
import numpy as np
import scipy as sc
import astropy.units as u
from astropy.io import fits
from astropy.time import Time
from astropy.table import Table, QTable
from astropy.coordinates import SkyCoord
from astroquery.heasarc import Heasarc
from SciServer import Authentication as auth
from heasoftpy import nustar, heatools, heasptools, heagen, ftools

In [ ]:
def NuSTAR(object_name, ObsIDs):
	# Step 0: Create folders for the data
	persistent_folder = os.path.join("/home/idies/workspace/Storage/", auth.getKeystoneUserWithToken(auth.getToken()).userName,
									 f"persistent/NuSTAR/{object_name}/")
	scratch_folder = os.path.join("/home/idies/workspace/Temporary/", auth.getKeystoneUserWithToken(auth.getToken()).userName,
								  f"scratch/NuSTAR/{object_name}/")
	Images = os.path.join(persistent_folder, "Images")
	Spectra = os.path.join(persistent_folder, "Spectra")
	Lightcurves = os.path.join(persistent_folder, "Light Curves")
	os.makedirs(Images, exist_ok=True)
	os.makedirs(Spectra, exist_ok=True)
	os.makedirs(Lightcurves, exist_ok=True)
	
	# ObsIDs selection
	Retrieved = (pyvo.regsearch(ivoid="ivo://nasa.heasarc/numaster")[0].get_service("conesearch").search(SkyCoord.from_name(object_name), verbosity=3))
	if len(ObsIDs) == 0:
		Table = Heasarc.query_region(position=SkyCoord.from_name(object_name), mission="numaster", radius=6 * u.arcminute,
									 fields="name,obsid,time,ra,dec,exposure_a,exposure_b,issue_flag,status")
		Filtered = Table[(Table["STATUS"] == "archived  ") & (Table["ISSUE_FLAG"] == 0) & (Table["EXPOSURE_A"] >= 1000) & (Table["EXPOSURE_B"] >= 1000)]
		Filtered.sort(keys="TIME")
		ObsIDs = Filtered["OBSID"].tolist()
	
	# ObsIDs iteration
	for obsid in ObsIDs:
		# Step 1: Data Accessing
		dlink = [res for res in Retrieved if (res["obsid"] == obsid)][0].getdatalink()
		link = [dl for dl in dlink if dl["content_type"] == "directory"][0]["access_url"]
		Source = ("/" + "/".join([part for part in link.split("FTP")[1].split("/") if part]) + "/")[:-1]
		Workdir = os.path.join(scratch_folder, obsid)
		os.makedirs(Workdir, exist_ok=True)
		os.chdir(Workdir)
		
		# Step 2: Data processing; images & spectra extraction
		nustar.nupipeline(instrument="ALL", obsmode="SCIENCE", indir=f"/home/idies/workspace/headata/FTP{Source}", steminputs=f"nu{obsid}", outdir=Workdir,
						  entrystage=1, exitstage=3, initseed=True, noprompt=True, allow_failure=True, cleancols=True, cleanup=True,
						  fpma_lcfile="NONE", fpmb_lcfile="NONE", fpma_bkglcfile="NONE", fpmb_bkglcfile="NONE", srcregionfile="DEFAULT",
						  fpma_phafile=f"nu{obsid}A_sr.pha", fpmb_phafile=f"nu{obsid}B_sr.pha", srcra="OBJECT", srcdec="OBJECT",
						  fpma_bkgphafile=f"nu{obsid}A01_bk.pha", fpmb_bkgphafile=f"nu{obsid}B01_bk.pha", bkgra="OBJECT", bkgdec="OBJECT",
						  fpma_imagefile=f"nu{obsid}A01_sk.img", fpmb_imagefile=f"nu{obsid}B01_sk.img", pntra="POINT", pntdec="POINT",
						  history=True, chatter=5, aberration=True, check_metsim_files=True, runmetrology=True, metflag=True, psdcal=True,
						  cleanflick=True, hpiterate=True, runcalcpha=True, runcalcpi=True, runflagevt=True, follow_sun=True, randomizecoordinator=True,
						  saamode="OPTIMIZED", tentacle=True, eliminatesource=True, tentacleregcut=True, nonulls=True, gtiscreen=True, evtscreen=True,
						  createattgti=True, createinstrgti=True, depthcut="NOMINAL", runsplitsc=False, timecut=True, createexpomap=True,
						  expovignflag=True, bkgextract=True, plotdevice="gif", correctlc=True, lcpsfflag=True, lcvignflag=True, psfflag=True,
						  arfvignflag=True, arfapstopflag=True, arfgrflag=True, arfdetabsflag=True, cutmaps=True, barycorr=False, runbackscale=True,
						  runmkarf=True, runmkrmf=True, cmprmf=True, arfmlicorr=True, clobber=True, verbose=False)
		heagen.barycorr(infile=f"nu{obsid}A01_cl.evt", outfile=f"nu{obsid}A01_cl.evt", orbitfiles=f"nu{obsid}A.attorb",
						clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
		heagen.barycorr(infile=f"nu{obsid}B01_cl.evt", outfile=f"nu{obsid}B01_cl.evt", orbitfiles=f"nu{obsid}B.attorb",
						clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
		# Step 3: Spectral regrouping & response file merging
		heasptools.ftgrouppha(infile=f'nu{obsid}A_sr.pha', backfile='%', respfile='%', outfile=f'nu{obsid}A01_sr.pha',
							  grouptype='optsnmin', groupscale=3, clobber=True)
		heasptools.ftgrouppha(infile=f'nu{obsid}B_sr.pha', backfile='%', respfile='%', outfile=f'nu{obsid}B01_sr.pha',
							  grouptype='optsnmin', groupscale=3, clobber=True)
		heasptools.ftmarfrmf(rmffile=f"nu{obsid}A01_sr.rmf", arffile=f"nu{obsid}A01_sr.arf", outfile=f"nu{obsid}A01_sr.rsp",
							 telescop="NuSTAR", instrume="FPMA", clobber=True)
		heasptools.ftmarfrmf(rmffile=f"nu{obsid}B01_sr.rmf", arffile=f"nu{obsid}B01_sr.arf", outfile=f"nu{obsid}B01_sr.rsp",
							 telescop="NuSTAR", instrume="FPMB", clobber=True)
		# Step 4: Merge region files and image files
		open(f'nu{obsid}A01.reg', mode='w').write(open(f'nu{obsid}A01_src.reg').read().removesuffix('\n') + "\n" +
												  open(f'nu{obsid}A01_bkg.reg').read().removesuffix('\n') + " # background")
		open(f'nu{obsid}B01.reg', mode='w').write(open(f'nu{obsid}B01_src.reg').read().removesuffix('\n') + "\n" +
												  open(f'nu{obsid}B01_bkg.reg').read().removesuffix('\n') + " # background")
		for i in [f'nu{obsid}A01_src.reg', f'nu{obsid}A01_bkg.reg', f'nu{obsid}B01_src.reg', f'nu{obsid}B01_bkg.reg']: os.remove(i) #clean-up
		heatools.ftpixcalc(outfile=f"nu{obsid}.img", expr='a+b', a=f"nu{obsid}A01_sk.img[PRIMARY]", b=f"nu{obsid}B01_sk.img[PRIMARY]", bitpix=-64, clobber=True)
		heatools.fthedit(infile=f'nu{obsid}A01_sk.img[PRIMARY]', operation='add', keyword='EXTNAME', value='IMAGE-A')
		heatools.fthedit(infile=f'nu{obsid}B01_sk.img[PRIMARY]', operation='add', keyword='EXTNAME', value='IMAGE-B')
		heatools.fthedit(infile=f'nu{obsid}A01_sk.img[GTI]',     operation='add', keyword='EXTNAME', value='GTI-A')
		heatools.fthedit(infile=f'nu{obsid}B01_sk.img[GTI]',     operation='add', keyword='EXTNAME', value='GTI-B')
		heatools.ftinsert(inhdu=f'nu{obsid}A01_sk.img[IMAGE-A]', infile=f"nu{obsid}.img[PRIMARY]", outfile=f"nu{obsid}.img", after=True, clobber=True)
		heatools.ftinsert(inhdu=f'nu{obsid}B01_sk.img[IMAGE-B]', infile=f"nu{obsid}.img[IMAGE-A]", outfile=f"nu{obsid}.img", after=True, clobber=True)
		heatools.ftinsert(inhdu=f'nu{obsid}A01_sk.img[GTI-A]',   infile=f"nu{obsid}.img[IMAGE-B]", outfile=f"nu{obsid}.img", after=True, clobber=True)
		heatools.ftinsert(inhdu=f'nu{obsid}B01_sk.img[GTI-B]',   infile=f"nu{obsid}.img[GTI-A]",   outfile=f"nu{obsid}.img", after=True, clobber=True)
		
		# Step 5: Move images and region file to Persistent folder
		shutil.copyfile(os.path.join(Workdir, f'nu{obsid}A01.reg'),    os.path.join(Images, f'nu{obsid}.reg'))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}.img"),       os.path.join(Images, f"nu{obsid}.img"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_im.gif"), os.path.join(Images, f"nu{obsid}A01_im.gif"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_im.gif"), os.path.join(Images, f"nu{obsid}B01_im.gif"))
		# Step 6: Move spectra files to Persistent folder
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_sr.pha"), os.path.join(Spectra, f"nu{obsid}A01_sr.pha"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_sr.pha"), os.path.join(Spectra, f"nu{obsid}B01_sr.pha"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_bk.pha"), os.path.join(Spectra, f"nu{obsid}A01_bk.pha"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_bk.pha"), os.path.join(Spectra, f"nu{obsid}B01_bk.pha"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_ph.gif"), os.path.join(Spectra, f"nu{obsid}A01_ph.gif"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_ph.gif"), os.path.join(Spectra, f"nu{obsid}B01_ph.gif"))
		# Step 7: Move ARF, RMF, & RSP files to Persistent folder
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_sr.arf"), os.path.join(Spectra, f"nu{obsid}A01_sr.arf"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_sr.arf"), os.path.join(Spectra, f"nu{obsid}B01_sr.arf"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_sr.rmf"), os.path.join(Spectra, f"nu{obsid}A01_sr.rmf"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_sr.rmf"), os.path.join(Spectra, f"nu{obsid}B01_sr.rmf"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}A01_sr.rsp"), os.path.join(Spectra, f"nu{obsid}A01_sr.rsp"))
		shutil.copyfile(os.path.join(Workdir, f"nu{obsid}B01_sr.rsp"), os.path.join(Spectra, f"nu{obsid}B01_sr.rsp"))
		
		# Step 8: Multi-band light curve extraction iteration
		PHA = np.array([3, 5, 7, 12, 30, 50, 78])
		for i in range(len(PHA)):
			if i + 1 < len(PHA): PI_min, PI_max = (PHA[i]-1.6)/0.04, (PHA[i+1]-1.6)/0.04
			else:                PI_min, PI_max = (PHA[0]-1.6)/0.04, (PHA[-1]-1.6)/0.04  # Full energy band
			# Conversion from PI channel to spectral energy
			E_min, E_max = int(0.04*PI_min+1.6), int(0.04*PI_max+1.6)
			# Step 8A: FPMA Light curve extraction
			nustar.nuproducts(indir=Workdir, instrument="FPMA", steminputs=f"nu{obsid}", outdir=Workdir, stemout=f"nu{obsid}A01_{E_min}-{E_max}",
							  srcregionfile=f"nu{obsid}A01_src.reg", bkgregionfile=f"nu{obsid}A01_bkg.reg", bkgextract=True, binsize=100,
							  lcfile=f"nu{obsid}A01_{E_min}-{E_max}_src.fits", bkglcfile=f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits", noprompt=True,
							  phafile="NONE", bkgphafile="NONE", imagefile="NONE", plotdevice="gif", pilow=int(PI_min), pihigh=int(PI_max),
							  aberration=True, extended=False, cutmaps=True, barycorr=False, usrgtibarycorr=True, write_baryevtfile=True, correctlc=True,
							  lcpsfflag=True, lcvignflag=True, psfflag=True, vignflag=True, apstopflag=True, grflag=True, detabsflag=True,
							  runmkarf=True, runmkrmf=True, cmprmf=True, runbackscale=True, cleanup=True, clobber=True, history=True, verbose=False)
			ftools.lcmath(infile=f"nu{obsid}A01_{E_min}-{E_max}_src.fits", bgfile=f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits",
						  outfile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", multi=1, multb=1, docor=True, err_mode=1)
			heagen.barycorr(infile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", outfile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", orbitfiles=f"nu{obsid}A.attorb",
							clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
			# Step 8B: FPMB Light curve extraction
			nustar.nuproducts(indir=Workdir, instrument="FPMB", steminputs=f"nu{obsid}", outdir=Workdir,stemout=f"nu{obsid}B01_{E_min}-{E_max}",
							  srcregionfile=f"nu{obsid}B01_src.reg", bkgregionfile=f"nu{obsid}B01_bkg.reg", bkgextract=True, binsize=100,
							  lcfile=f"nu{obsid}B01_{E_min}-{E_max}_src.fits", bkglcfile=f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits", noprompt=True,
							  phafile="NONE", bkgphafile="NONE", imagefile="NONE", plotdevice="gif", pilow=int(PI_min), pihigh=int(PI_max),
							  aberration=True, extended=False, cutmaps=True, barycorr=False, usrgtibarycorr=True, write_baryevtfile=True, correctlc=True,
							  lcpsfflag=True, lcvignflag=True, psfflag=True, vignflag=True, apstopflag=True, grflag=True, detabsflag=True,
							  runmkarf=True, runmkrmf=True, cmprmf=True, runbackscale=True, cleanup=True, clobber=True, history=True, verbose=False)
			ftools.lcmath(infile=f"nu{obsid}B01_{E_min}-{E_max}_src.fits", bgfile=f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits",
						  outfile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", multi=1, multb=1, docor=True, err_mode=1)
			heagen.barycorr(infile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", outfile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", orbitfiles=f"nu{obsid}B.attorb",
							clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
		
		# Step 9: light curves merging
		fits.setval(filename=f"LightCurve-A_{obsid}_3-78.fits", extname='RATE', keyword='EXTNAME', value="3-78")
		fits.setval(filename=f"LightCurve-B_{obsid}_3-78.fits", extname='RATE', keyword='EXTNAME', value="3-78")
		fits.setval(filename=f"LightCurve-A_{obsid}_3-5.fits", extname='RATE', keyword='EXTNAME', value="3-5")
		fits.setval(filename=f"LightCurve-B_{obsid}_3-5.fits", extname='RATE', keyword='EXTNAME', value="3-5")
		heatools.ftinsert(inhdu=f'LightCurve-A_{obsid}_3-5.fits[1]', infile=f'LightCurve-A_{obsid}_3-78.fits[1]',
						  outfile=f"LightCurve-FPMA_{obsid}.fits", after=True, clobber=True)
		heatools.ftinsert(inhdu=f'LightCurve-B_{obsid}_3-5.fits[1]', infile=f'LightCurve-B_{obsid}_3-78.fits[1]',
						  outfile=f"LightCurve-FPMB_{obsid}.fits", after=True, clobber=True)
		i = 2
		for E_min,E_max in zip(PHA[1:], PHA[2:]):
			fits.setval(filename=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", extname='RATE', keyword='EXTNAME', value=f"{E_min}-{E_max}")
			fits.setval(filename=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", extname='RATE', keyword='EXTNAME', value=f"{E_min}-{E_max}")
			heatools.ftinsert(inhdu=f'LightCurve-A_{obsid}_{E_min}-{E_max}.fits[1]', infile=f"LightCurve-FPMA_{obsid}.fits[{i}]",
							  outfile=f"LightCurve-FPMA_{obsid}.fits", after=True, clobber=True)
			heatools.ftinsert(inhdu=f'LightCurve-B_{obsid}_{E_min}-{E_max}.fits[1]', infile=f"LightCurve-FPMB_{obsid}.fits[{i}]",
							  outfile=f"LightCurve-FPMB_{obsid}.fits", after=True, clobber=True)
			i += 1
		# Move (background-subtracted) light curve to Persistent folder
		shutil.copyfile(os.path.join(Workdir, f"LightCurve-FPMA_{obsid}.fits"), os.path.join(Lightcurves, f"LightCurve-FPMA_{obsid}.fits"))
		shutil.copyfile(os.path.join(Workdir, f"LightCurve-FPMB_{obsid}.fits"), os.path.join(Lightcurves, f"LightCurve-FPMB_{obsid}.fits"))

		# Step 10: Corrected, binned, reduced FPMA source event list
		eventlists = []
		Energy_bands = np.geomspace(3,78,15)
		for E_min,E_max in zip(Energy_bands, Energy_bands[1:]):
			PI_min, PI_max = np.rint((E_min-1.6)/0.04), np.rint((E_max-1.6)/0.04)
			nustar.nuproducts(indir=Workdir, instrument="FPMA", steminputs=f"nu{obsid}", outdir=Workdir, stemout=f"nu{obsid}A01_{E_min}-{E_max}",
							  srcregionfile=f"nu{obsid}A01_src.reg", bkgregionfile=f"nu{obsid}A01_bkg.reg", bkgextract=True, binsize=100,
							  lcfile=f"nu{obsid}A01_{E_min}-{E_max}_src.fits", bkglcfile=f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits", noprompt=True,
							  phafile="NONE", bkgphafile="NONE", imagefile="NONE", plotdevice="gif", pilow=int(PI_min), pihigh=int(PI_max),
							  aberration=True, extended=False, cutmaps=True, barycorr=False, usrgtibarycorr=True, write_baryevtfile=True, correctlc=True,
							  lcpsfflag=True, lcvignflag=True, psfflag=True, vignflag=True, apstopflag=True, grflag=True, detabsflag=True,
							  runmkarf=True, runmkrmf=True, cmprmf=True, runbackscale=True, cleanup=True, clobber=True, history=True, verbose=False)
			ftools.lcmath(infile=f"nu{obsid}A01_{E_min}-{E_max}_src.fits", bgfile=f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits",
						  outfile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", multi=1, multb=1, docor=True, err_mode=1)
			heagen.barycorr(infile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", outfile=f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits", orbitfiles=f"nu{obsid}A.attorb",
							clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
			filename = f"LightCurve-A_{obsid}_{E_min}-{E_max}.fits"
			Header = fits.getheader(filename, ext=1)
			lc = QTable.read(filename, format='fits', hdu=1)
			lc = lc[lc['FRACEXP']!=0]  # Delete rows with masked values
			evl = stingray.EventList.from_lc(stingray.Lightcurve(time=lc['TIME']+Header['TSTART']*u.s, counts=lc['RATE'], err=lc['ERROR'],
																 bg_counts=QTable.read(f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits")['RATE'],
																 frac_exp=lc['FRACEXP'], input_counts=False, err_dist='poisson',
																 gti=np.array(QTable.read(filename, format='fits', hdu='GTI').as_array().tolist()),
																 dt=Header['TIMEDEL'], header=Header, mission='NuSTAR', instr='FPMA'))
			evl.energy   = [sc.stats.gmean([E_min,E_max])]*len(evl)
			evl.header   = Header
			evl.dt       = Header['TIMEDEL']
			evl.mjdref   = Header['MJDREFI'] + Header['MJDREFF']
			evl.timeref  = 'SOLARSYSTEM'
			evl.timesys  = 'TDB'
			evl.rmf_file = f'Spectra/nu{obsid}A01_sr.rsp'
			evl.mission  = "NuSTAR"
			evl.instr    = "FPMA"
			evl.detector_id = "NuSTAR"
			evl.high_precision = True
			eventlists.append(evl)
			os.remove(f"nu{obsid}A01_{E_min}-{E_max}_src.fits")
			os.remove(f"nu{obsid}A01_{E_min}-{E_max}_bkg.fits")
			os.remove(filename)
		EVL = eventlists[0].join(other=eventlists[1:], strategy='union')
		EVL.energy = np.array(EVL.energy)
		EVL.write(filename=f'EVL-FPMA_{obsid}.fits', fmt='fits')
		HDUL = fits.open(f'nu{obsid}A01_cl.evt').copy()
		HDUL['PRIMARY'].header.remove(keyword='HISTORY', ignore_missing=True, remove_all=True)
		HDUL['EVENTS'].data = fits.getdata(f'EVL-FPMA_{obsid}.fits', ext=1)
		HDUL['EVENTS'].update_header()
		fits.HDUList(hdus=[HDUL['PRIMARY'], HDUL['EVENTS'], HDUL['GTI']]).writeto(fileobj=os.path.join(persistent_folder, f'EVL-FPMA_{obsid}.fits'),
																				  output_verify='fix+warn', overwrite=True, checksum=True)
		
		# Step 11: Corrected, binned, reduced FPMB source event list
		eventlists = []
		Energy_bands = np.geomspace(3,78,15)
		for E_min,E_max in zip(Energy_bands, Energy_bands[1:]):
			PI_min, PI_max = np.rint((E_min-1.6)/0.04), np.rint((E_max-1.6)/0.04)
			nustar.nuproducts(indir=Workdir, instrument="FPMB", steminputs=f"nu{obsid}", outdir=Workdir,stemout=f"nu{obsid}B01_{E_min}-{E_max}",
							  srcregionfile=f"nu{obsid}B01_src.reg", bkgregionfile=f"nu{obsid}B01_bkg.reg", bkgextract=True, binsize=100,
							  lcfile=f"nu{obsid}B01_{E_min}-{E_max}_src.fits", bkglcfile=f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits", noprompt=True,
							  phafile="NONE", bkgphafile="NONE", imagefile="NONE", plotdevice="gif", pilow=int(PI_min), pihigh=int(PI_max),
							  aberration=True, extended=False, cutmaps=True, barycorr=False, usrgtibarycorr=True, write_baryevtfile=True, correctlc=True,
							  lcpsfflag=True, lcvignflag=True, psfflag=True, vignflag=True, apstopflag=True, grflag=True, detabsflag=True,
							  runmkarf=True, runmkrmf=True, cmprmf=True, runbackscale=True, cleanup=True, clobber=True, history=True, verbose=False)
			ftools.lcmath(infile=f"nu{obsid}B01_{E_min}-{E_max}_src.fits", bgfile=f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits",
						  outfile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", multi=1, multb=1, docor=True, err_mode=1)
			heagen.barycorr(infile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", outfile=f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits", orbitfiles=f"nu{obsid}B.attorb",
							clockfile='CALDB', refframe='ICRS', ephem='DEFAULT', barytime=False, clobber=True, history=True)
			filename = f"LightCurve-B_{obsid}_{E_min}-{E_max}.fits"
			Header = fits.getheader(filename, ext=1)
			lc = QTable.read(filename, format='fits', hdu=1)
			lc = lc[lc['FRACEXP']!=0]  # Delete rows with masked values
			evl = stingray.EventList.from_lc(stingray.Lightcurve(time=lc['TIME']+Header['TSTART']*u.s, counts=lc['RATE'], err=lc['ERROR'],
																 bg_counts=QTable.read(f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits")['RATE'],
																 frac_exp=lc['FRACEXP'], input_counts=False, err_dist='poisson',
																 gti=np.array(QTable.read(filename, format='fits', hdu='GTI').as_array().tolist()),
																 dt=Header['TIMEDEL'], header=Header, mission='NuSTAR', instr='FPMB'))
			evl.energy   = [sc.stats.gmean([E_min,E_max])]*len(evl)
			evl.header   = Header
			evl.dt       = Header['TIMEDEL']
			evl.mjdref   = Header['MJDREFI'] + Header['MJDREFF']
			evl.timeref  = 'SOLARSYSTEM'
			evl.timesys  = 'TDB'
			evl.rmf_file = f'Spectra/nu{obsid}B01_sr.rsp'
			evl.mission  = "NuSTAR"
			evl.instr    = "FPMB"
			evl.detector_id = "NuSTAR"
			evl.high_precision = True
			eventlists.append(evl)
			os.remove(f"nu{obsid}B01_{E_min}-{E_max}_src.fits")
			os.remove(f"nu{obsid}B01_{E_min}-{E_max}_bkg.fits")
			os.remove(filename)
		EVL = eventlists[0].join(other=eventlists[1:], strategy='union')
		EVL.energy = np.array(EVL.energy)
		EVL.write(filename=f'EVL-FPMB_{obsid}.fits', fmt='fits')
		HDUL = fits.open(f'nu{obsid}B01_cl.evt').copy()
		HDUL['PRIMARY'].header.remove(keyword='HISTORY', ignore_missing=True, remove_all=True)
		HDUL['EVENTS'].data = fits.getdata(f'EVL-FPMB_{obsid}.fits', ext=1)
		HDUL['EVENTS'].update_header()
		fits.HDUList(hdus=[HDUL['PRIMARY'], HDUL['EVENTS'], HDUL['GTI']]).writeto(fileobj=os.path.join(persistent_folder, f'EVL-FPMB_{obsid}.fits'),
																				  output_verify='fix+warn', overwrite=True, checksum=True)
	
	print("-------------------------------- FINISHED! --------------------------------")

In [ ]:
#NuSTAR(object_name="NGC4051", ObsIDs=['60160466002'])
#NuSTAR(object_name="NGC3516", ObsIDs=['80901608002','60160001002','60801025002','60801025004'])
#NuSTAR(object_name="Mrk509", ObsIDs=[])